# Exploration Article 9 RGPD

Ce notebook lit les exports produits par `python src/run_article9_scan.py` et donne quelques vues rapides pour explorer les resultats.

Le scan documentaire est offline-by-default. Les telechargements de modeles doivent passer par `python src/run_article9_prepare_resources.py`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent

repo_root

In [ ]:
# Decommente cette cellule pour relancer le scan depuis le notebook.
# import subprocess
# import sys
# subprocess.run([sys.executable, str(repo_root / "src" / "run_article9_scan.py")], check=True)

processed_dir = repo_root / "data" / "processed"
findings_path = processed_dir / "article9_findings.csv"
summary_path = processed_dir / "article9_summary.csv"
audit_path = processed_dir / "article9_audit_log.json"

if not findings_path.exists() or not summary_path.exists():
    raise FileNotFoundError(
        "Les fichiers de sortie sont absents. Lance d'abord : python src/run_article9_scan.py"
    )

findings = pd.read_csv(findings_path)
summary = pd.read_csv(summary_path)

findings.head(), summary.head()

In [ ]:
summary.sort_values(["finding_count", "max_score"], ascending=[False, False])

In [ ]:
entity_counts = findings.groupby("category_label").size().sort_values(ascending=False)
entity_counts

In [ ]:
ax = entity_counts.plot(kind="bar", figsize=(10, 4), color="#355C7D")
ax.set_title("Nombre de detections par entite")
ax.set_xlabel("Categorie")
ax.set_ylabel("Detections")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

In [ ]:
file_counts = findings.groupby("file_name").size().sort_values(ascending=False)
ax = file_counts.plot(kind="bar", figsize=(12, 4), color="#C06C84")
ax.set_title("Nombre de detections par fichier")
ax.set_xlabel("Fichier")
ax.set_ylabel("Detections")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()

In [ ]:
pivot = (
    findings.pivot_table(
        index="file_name",
        columns="category_label",
        values="matched_text",
        aggfunc="count",
        fill_value=0,
    )
    .sort_index()
)
pivot.style.background_gradient(cmap="OrRd")

In [ ]:
focus_file = summary.sort_values("finding_count", ascending=False).iloc[0]["file_name"]
focus = findings.loc[findings["file_name"] == focus_file].sort_values(["page_number", "category_score", "raw_score"], ascending=[True, False, False])
focus[["file_name", "page_number", "category_label", "decision", "method", "matched_text", "matched_variant", "category_score", "context_hits", "excerpt", "explanation"]]

In [ ]:
decision_counts = findings.groupby("decision").size().sort_values(ascending=False)
decision_counts

In [ ]:
ax = decision_counts.plot(kind="bar", figsize=(6, 4), color="#6C5B7B")
ax.set_title("Repartition des decisions")
ax.set_xlabel("Decision")
ax.set_ylabel("Detections")
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
method_counts = findings.groupby("method").size().sort_values(ascending=False)
method_counts

In [ ]:
ax = method_counts.plot(kind="bar", figsize=(8, 4), color="#F67280")
ax.set_title("Contribution par methode")
ax.set_xlabel("Methode")
ax.set_ylabel("Detections")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()

In [ ]:
findings.sort_values(["category_score", "raw_score"], ascending=[False, False])[["file_name", "category_label", "decision", "method", "matched_text", "matched_variant", "category_score", "excerpt", "explanation"]].head(20)

In [ ]:
if audit_path.exists():
    audit_log = json.loads(audit_path.read_text(encoding="utf-8"))
    len(audit_log)
else:
    audit_log = []
    print("Audit log absent. Relance le pipeline pour generer article9_audit_log.json")

In [ ]:
audit_overview = pd.DataFrame(
    [
        {
            "text_id": item["text_id"],
            "file_name": item.get("metadata", {}).get("file_name", item["text_id"]),
            "chunk_id": item.get("metadata", {}).get("chunk_id", ""),
            "page_number": item.get("metadata", {}).get("page_number"),
            "decision_log": " | ".join(item.get("decision_log", [])),
            "active_categories": ", ".join(
                category["category_key"]
                for category in item.get("categories", [])
                if category.get("decision") != "clear"
            ),
        }
        for item in audit_log
    ]
)
audit_overview.head(20)

In [ ]:
selected = next(
    (
        item for item in audit_log
        if any(category.get("decision") in {"review", "alert"} for category in item.get("categories", []))
    ),
    audit_log[0] if audit_log else None,
)
selected

In [ ]:
decision_log_rows = []
for item in audit_log:
    for entry in item.get("decision_log", []):
        decision_log_rows.append(
            {
                "file_name": item.get("metadata", {}).get("file_name", item["text_id"]),
                "chunk_id": item.get("metadata", {}).get("chunk_id", ""),
                "decision_log_entry": entry,
            }
        )

decision_log_df = pd.DataFrame(decision_log_rows)
decision_log_df.head(20)

In [ ]:
decision_log_df.groupby("decision_log_entry").size().sort_values(ascending=False)

In [ ]:
alerts_only = findings.loc[findings["decision"] == "alert"].sort_values(["category_score", "raw_score"], ascending=[False, False])
alerts_only[["file_name", "page_number", "category_label", "method", "matched_text", "matched_variant", "category_score", "excerpt", "explanation"]]

In [ ]:
possible_false_positives = findings.loc[
    (findings["decision"] == "review")
    & (findings["method"] == "fuzzy")
].sort_values(["category_score", "raw_score"], ascending=[False, False])

possible_false_positives[["file_name", "category_label", "matched_text", "matched_variant", "raw_score", "category_score", "excerpt", "explanation"]].head(30)